# Milestone 4 — Inference profiling (CPU-oriented)

This notebook measures **preprocessing**, **embedding**, and **scoring** latency for the same path used in production (`src.inference.infer_pair`), plus **end-to-end** time and **batch-size sensitivity** (processing multiple pairs in one timed batch).

**Step 1 (this notebook):** environment + paths + methodology. Run the cells top to bottom after `pip install -r requirements.txt` (optional: `jupyter` / `ipykernel` if your editor needs them).

## 1. Methodology (fill in when you finalize the report)

| Item | Value |
|------|--------|
| Timer | `time.perf_counter()` |
| Warm-up | Separate warm-up passes before timed runs (model + I/O cache) |
| Repeats per stage | Tune `N_REPEATS` below |
| Batch sizes | Tune `BATCH_SIZES` — here “batch” = number of pairs processed sequentially in one timed loop |
| Hardware | Record output of the environment cell when exporting the profiling report |

**Scoring bucket for Milestone wording:** cosine similarity plus threshold + confidence (remainder after embedding) — see `combined_scoring_ms` in the summary tables.

In [1]:
from __future__ import annotations

import csv
import platform
import sys
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import pandas as pd

# Project root = parent of reports/
NOTEBOOK_DIR = Path.cwd().resolve()
REPORTS_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "reports" else NOTEBOOK_DIR / "reports"
PROJECT_ROOT = REPORTS_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Config
from src.inference import infer_pair

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
PAIRS_CSV = PROJECT_ROOT / "outputs" / "pairs" / "test_pairs.csv"

N_WARMUP = 3
N_REPEATS = 30  # increase for tighter confidence intervals
BATCH_SIZES = [1, 2, 4, 8, 16]

print("PROJECT_ROOT", PROJECT_ROOT)
print("CONFIG_PATH", CONFIG_PATH)
print("PAIRS_CSV", PAIRS_CSV)

PROJECT_ROOT /Users/nandanprince/Desktop/test_face-ID/FaceID_Verification
CONFIG_PATH /Users/nandanprince/Desktop/test_face-ID/FaceID_Verification/configs/default.yaml
PAIRS_CSV /Users/nandanprince/Desktop/test_face-ID/FaceID_Verification/outputs/pairs/test_pairs.csv


## 2. Measurement environment

Copy this block into your written **Profiling report** when you submit Milestone 4.

In [2]:
def torch_device_summary() -> str:
    try:
        import torch

        cuda = torch.cuda.is_available()
        return f"torch={torch.__version__} cuda_available={cuda}"
    except Exception as exc:  # pragma: no cover
        return f"torch not importable ({exc})"


print("python", sys.version.replace("\n", " "))
print("platform", platform.platform())
print("processor", platform.processor())
print(torch_device_summary())

config = Config.from_file(CONFIG_PATH)
emb = config._config.get("embedding", {})
print("embedding.backend", emb.get("backend"))
print("embedding.device", emb.get("device"))
print("embedding.preprocess_size", emb.get("preprocess_size"))

python 3.11.0 (v3.11.0:deaf509e8f, Oct 24 2022, 14:43:23) [Clang 13.0.0 (clang-1300.0.29.30)]
platform macOS-15.2-arm64-arm-64bit
processor arm
torch=2.9.1 cuda_available=False
embedding.backend inceptionresnetv1
embedding.device cpu
embedding.preprocess_size (160, 160)


## 3. Load sample pairs

Uses `outputs/pairs/test_pairs.csv` when present (run the Milestone 1–2 pairing pipeline if missing). Falls back to two paths used in tests.

In [3]:
def load_pairs(max_pairs: int) -> list[tuple[str, str]]:
    pairs: list[tuple[str, str]] = []
    if PAIRS_CSV.is_file():
        with PAIRS_CSV.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                pairs.append((row["left_path"], row["right_path"]))
                if len(pairs) >= max_pairs:
                    break
    if not pairs:
        fallback = [
            ("data/lfw/images/A/000001.jpg", "data/lfw/images/B/000001.jpg"),
        ]
        pairs = fallback
    return pairs[:max_pairs]


MAX_FOR_BATCH = max(BATCH_SIZES)
sample_pairs = load_pairs(max(MAX_FOR_BATCH, 32))
left0, right0 = sample_pairs[0]
print("num sample pairs", len(sample_pairs))
print("first pair", left0, right0)

num sample pairs 32
first pair data/lfw/images/Renee_Zellweger/004147.jpg data/lfw/images/Renee_Zellweger/013225.jpg


## 4. Warm-up

Ensures the embedding model is loaded and filesystem caches are warm before timed sections.

In [4]:
for _ in range(N_WARMUP):
    infer_pair(left0, right0, config)
print("warm-up complete")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nandanprince/Desktop/test_face-ID/FaceID_Verification/reports/data/lfw/images/Renee_Zellweger/004147.jpg'

## 5. Per-stage latency (single pair, repeated)

Uses `infer_pair` so numbers match the CLI. Columns follow `stage_latency_ms` with an added **`combined_scoring_ms`** = similarity + threshold + confidence.

In [5]:
rows: list[dict[str, Any]] = []
for _ in range(N_REPEATS):
    out = infer_pair(left0, right0, config)
    st = out["stage_latency_ms"]
    rows.append(
        {
            "preprocess_ms": st["preprocessing"],
            "embedding_ms": st["embedding_generation"],
            "similarity_ms": st["similarity_scoring"],
            "threshold_ms": st["threshold_decision"],
            "confidence_ms": st["confidence_computation"],
            "total_ms": st["total"],
        }
    )

df = pd.DataFrame(rows)
df["combined_scoring_ms"] = df["similarity_ms"] + df["threshold_ms"] + df["confidence_ms"]
summary = pd.DataFrame(
    {
        "mean_ms": df.mean(),
        "std_ms": df.std(ddof=1),
        "p50_ms": df.median(),
    }
)
summary

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nandanprince/Desktop/test_face-ID/FaceID_Verification/reports/data/lfw/images/Renee_Zellweger/004147.jpg'

In [6]:
stage_means = {
    "preprocessing": df["preprocess_ms"].mean(),
    "embedding": df["embedding_ms"].mean(),
    "scoring (combined)": df["combined_scoring_ms"].mean(),
    "end-to-end total": df["total_ms"].mean(),
}
pd.Series(stage_means, name="mean_latency_ms")

NameError: name 'df' is not defined

## 6. Batch-size sensitivity

For each batch size `B`, we time **one block** that runs `B` pairs **sequentially** (same as today’s CLI batch: no fused multi-image forward pass). Throughput = `B / elapsed`.

In [7]:
batch_rows: list[dict[str, Any]] = []
for b in BATCH_SIZES:
    chunk = sample_pairs[:b]
    t0 = perf_counter()
    for left_p, right_p in chunk:
        infer_pair(left_p, right_p, config)
    elapsed = perf_counter() - t0
    batch_rows.append(
        {
            "batch_size": b,
            "elapsed_s": elapsed,
            "pairs_per_s": b / elapsed if elapsed > 0 else float("nan"),
            "ms_per_pair_mean": (elapsed / b) * 1000.0,
        }
    )

batch_df = pd.DataFrame(batch_rows)
batch_df

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nandanprince/Desktop/test_face-ID/FaceID_Verification/reports/data/lfw/images/Renee_Zellweger/004147.jpg'

## 7. Plots (optional export for `reports/`)

Requires `matplotlib` (already in project `requirements.txt`).

In [8]:
import matplotlib.pyplot as plt

labels = list(stage_means.keys())[:-1]  # drop total for stacked-style bar comparison
values = [stage_means[k] for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(labels, values, color=["#4477AA", "#EE6677", "#228833"])
axes[0].set_ylabel("Mean latency (ms)")
axes[0].set_title("Per-stage mean (single pair)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].plot(batch_df["batch_size"], batch_df["pairs_per_s"], marker="o")
axes[1].set_xlabel("Batch size (pairs per timed block)")
axes[1].set_ylabel("Throughput (pairs / s)")
axes[1].set_title("Batch sensitivity (sequential pairs)")

plt.tight_layout()
plt.show()

NameError: name 'stage_means' is not defined

## Next steps (later cells you can add)

- Optional **GPU** section: set `embedding.device: "auto"` in a copy of the config (or override YAML) and repeat sections 4–6; label clearly as supplemental.
- Export **CSV** summaries to `reports/evidence/profiling/` for the grader.
- Increase `N_REPEATS` and document hardware in your Milestone 4 write-up.

**Step 2:** Run all cells and paste the environment printout + tables into your profiling report. **Step 3:** optional GPU and artifact export.